In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/models/justineleonuro/deepseek-math/transformers/deepseek-math-7b-instruct/1/deepseek-math-7bin-p100/model.safetensors.index.json
/kaggle/input/models/justineleonuro/deepseek-math/transformers/deepseek-math-7b-instruct/1/deepseek-math-7bin-p100/model-00003-of-00003.safetensors
/kaggle/input/models/justineleonuro/deepseek-math/transformers/deepseek-math-7b-instruct/1/deepseek-math-7bin-p100/config.json
/kaggle/input/models/justineleonuro/deepseek-math/transformers/deepseek-math-7b-instruct/1/deepseek-math-7bin-p100/tokenizer.json
/kaggle/input/models/justineleonuro/deepseek-math/transformers/deepseek-math-7b-instruct/1/deepseek-math-7bin-p100/model-00001-of-00003.safetensors
/kaggle/input/models/justineleonuro/deepseek-math/transformers/deepseek-math-7b-instruct/1/deepseek-math-7bin-p100/tokenizer_config.json
/kaggle/input/models/justineleonuro/deepseek-math/transformers/deepseek-math-7b-instruct/1/deepseek-math-7bin-p100/model-00002-of-00003.safetensors
/kaggle/input/mode

In [2]:
import os
import re
import sympy as sp
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
COMPETITION_DIR_CANDIDATES = [
    "/kaggle/input/ai-mathematical-olympiad-progress-prize-3",
    "/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3",
]

ANSWER_MOD = 100000
DEBUG = True

In [4]:
def resolve_competition_dir():
    for path in COMPETITION_DIR_CANDIDATES:
        if os.path.exists(path):
            return path
    raise FileNotFoundError(
        "Competition input directory was not found. "
        "Please check the actual path under /kaggle/input."
    )

competition_dir = resolve_competition_dir()

if DEBUG:
    print("Competition directory:", competition_dir)
    print("Files:", os.listdir(competition_dir))

test_path = os.path.join(competition_dir, "test.csv")
sample_submission_path = os.path.join(competition_dir, "sample_submission.csv")

Competition directory: /
Files: ['bin', 'var', 'mnt', 'etc', 'root', 'home', 'tmp', 'lib', 'sbin', 'proc', 'lib64', 'srv', 'boot', 'opt', 'sys', 'media', 'lib32', 'usr', 'run', 'libx32', 'dev', 'kaggle', '.dockerenv', '.jupyter', 'requirements.txt', 'colab_requirements.txt', 'kaggle_requirements.txt', 'datalab', 'tools', 'content', 'python-apt', 'python-apt.tar.xz', 'NGC-DL-CONTAINER-LICENSE', 'cuda-keyring_1.1-1_all.deb']


In [5]:
test_df = pd.read_csv(test_path)

if DEBUG:
    print("test.csv shape:", test_df.shape)
    print(test_df.head())

FileNotFoundError: [Errno 2] No such file or directory: '/test.csv'

In [ ]:
def extract_boxed_number(text: str):
    if not isinstance(text, str):
        return None
    
    match = re.search(r'\\boxed\{(-?\d+)\}', text)
    if match:
        return int(match.group(1)) % ANSWER_MOD
    
    return None

def solve_with_sympy_if_easy(problem: str):
    if not isinstance(problem, str):
        return 0
    
    # Very conservative baseline:
    # only solve extremely simple one-variable polynomial equations if detected
    try:
        cleaned = problem.strip()

        # Example pattern:
        # "Solve x^2 - 5x + 6 = 0"
        simple_eq_match = re.search(r'([xX][\s\S]*?)=\s*0', cleaned)
        if simple_eq_match and len(cleaned) < 200:
            x = sp.symbols('x')
            expr_text = simple_eq_match.group(1)

            expr_text = expr_text.replace("^", "**")
            expr_text = re.sub(r'(\d)([xX])', r'\1*\2', expr_text)
            expr_text = expr_text.replace("X", "x")

            expr = sp.sympify(expr_text)
            roots = sp.solve(sp.Eq(expr, 0), x)

            integer_roots = []
            for r in roots:
                if r.is_integer:
                    integer_roots.append(int(r))

            if integer_roots:
                return min(integer_roots) % ANSWER_MOD

    except Exception:
        pass

    # Safe fallback answer
    return 0

def predict_answer(problem: str):
    # This is a placeholder baseline.
    # Replace this function with a stronger solver later.
    return solve_with_sympy_if_easy(problem)


In [ ]:
submission_df = pd.DataFrame()
submission_df["id"] = test_df["id"]

problem_column = None
for candidate in ["problem", "question", "prompt"]:
    if candidate in test_df.columns:
        problem_column = candidate
        break

if problem_column is None:
    raise ValueError(
        "No problem text column was found. "
        "Expected one of: problem, question, prompt"
    )

submission_df["answer"] = test_df[problem_column].apply(predict_answer).astype(int) % ANSWER_MOD
submission_df["answer"] = submission_df["answer"].clip(0, ANSWER_MOD - 1)

In [ ]:
output_path = "/kaggle/working/submission.csv"
submission_df.to_csv(output_path, index=False)

print("Saved submission file to:", output_path)
print(submission_df.head())